In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
data.head()

In [ ]:
data = data.values #shape (m, 785) first one is the label or answer, remaining 784 are the pixels (28x28)
m,n = data.shape #m is rows and n is columns
np.random.shuffle(data)

In [ ]:
X_all = data[:, 1:].T.astype(np.float32)
y_all = data[:, 0].astype(np.int64)

X_all /= 255.0 #normalizing pixel values from 0-255 to 0.0-1.0 (prevents softmax values from exploding)

#splitting into test and train sets
test_size = 1000
X_test = X_all[:, :test_size]
y_test = y_all[:test_size]
X_train = X_all[:, test_size:]
y_train = y_all[test_size:]

In [ ]:
print(X_train[:, 0].shape)

In [ ]:
def one_hot_encoder(Y, num_classes = 10):
    m = Y.size
    onehotY = np.zeros((num_classes, m))
    onehotY[Y.astype(int), np.arange(m)] = 1
    return onehotY
    
def softmax(Z):
    Z = Z - np.max(Z, axis=0, keepdims=True)
    expZ = np.exp(Z)
    return expZ / np.sum(expZ, axis=0, keepdims=True)
    
def ReLU(Z):
    return np.maximum(0, Z)
    
def d_ReLU(Z):
    return (Z>0).astype(float)
    
def init_params(nx, nh, ny): #He/kaiming initialization for weights (again prevents explosion, this time for gradients)
    W1 = np.random.randn(nh, nx) * np.sqrt(2.0/nx)
    b1 = np.zeros((nh, 1))
    W2 = np.random.randn(ny, nh) * np.sqrt(2.0/nh)
    b2 = np.zeros((ny, 1))

    return W1, b1, W2, b2

def compute_loss(A2, Y):
    m = Y.size
    onehotY = one_hot_encoder(Y, num_classes=A2.shape[0])

    log_probability = np.log(A2 + 1e-12)
    loss = -np.sum(onehotY * log_probability) / m

    return loss

def forward_propagation(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)

    return Z1, A1, Z2, A2

def back_propagation(Z1, A1, Z2, A2, W2, X, Y):
    m = Y.size
    onehotY = one_hot_encoder(Y, num_classes=A2.shape[0])
    dZ2 = A2 - onehotY
    dW2 = 1.0/m * dZ2.dot(A1.T)
    db2 = 1.0/m * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * d_ReLU(Z1)
    dW1 = 1.0/m * dZ1.dot(X.T)
    db1 = 1.0/m * np.sum(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 -= alpha*dW1
    b1 -= alpha*db1
    W2 -= alpha*dW2
    b2 -= alpha*db2

    return W1, b1, W2, b2

def get_predictions(A2):
    return np.argmax(A2, axis=0)

def get_accuracy(predictions, Y):
    return np.mean(predictions == Y)

In [ ]:
def train(X, Y, nh = 128, epochs = 10, batch_size = 64, alpha = 0.1, print_every = 1):
    nx = X.shape[0]
    ny = 10

    W1, b1, W2, b2 = init_params(nx, nh, ny)
    for epoch in range(epochs):
        perm = np.random.permutation(Y.size)
        X_shuffle = X[:, perm]
        Y_shuffle = Y[perm]
        for i in range(0, Y.size, batch_size):
            X_batch = X_shuffle[:, i:i+batch_size]
            Y_batch = Y_shuffle[i:i+batch_size]

            Z1, A1, Z2, A2 = forward_propagation(W1, b1, W2, b2, X_batch)
            dW1, db1, dW2, db2 = back_propagation(Z1, A1, Z2, A2, W2, X_batch, Y_batch)
            W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
    
        if((epoch+1) % print_every) == 0:
            Z1_all, A1_all, Z2_all, A2_all = forward_propagation(W1, b1, W2, b2, X)
            loss = compute_loss(A2_all, Y)
            predictions = get_predictions(A2_all)
            accuracy = get_accuracy(predictions, Y)
            print(f'Epoch {epoch+1}/{epochs}, loss = {loss:.4f}, training accuracy = {accuracy:.4f}')
            
    return W1, b1, W2, b2

In [ ]:
W1, b1, W2, b2 = train(X_train, y_train, nh=128, epochs=50, batch_size=128, alpha=0.001, print_every=5)

In [ ]:
Z1_test, A1_test, Z2_test, A2_test = forward_propagation(W1, b1, W2, b2, X_test)
test_predictions = get_predictions(A2_test)
test_accuracy = get_accuracy(test_predictions, y_test)

print(f'Test accuracy =  {test_accuracy:.4f}')

In [ ]:
def plot_image(index):
    image = X_train[:, index, None]
    label = y_train[index]
    _ , _ , _ , A2_test = forward_propagation(W1, b1, W2, b2, image)
    test_predictions = get_predictions(A2_test)
    print("Prediction: ", test_predictions)
    print("Label: ", label)
    image = image.reshape((28, 28)) * 255
    plt.gray()
    plt.imshow(image, interpolation='nearest')
    plt.show()

In [ ]:
plot_image(786)